# Lecture 1: Unit introduction

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from enn554.paths import data_dir, repo_root, package_root # this is the path to the data directory
import matplotlib.pyplot as plt
import seaborn as sns
dd = data_dir()

# Energy use in Australia
Data from [IEA](https://www.iea.org/countries/australia/energy-mix#how-is-energy-used-in-australia)

In [ ]:


sns.set_theme(style="whitegrid")

# Publication-style defaults
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})

# Explicit column order: fossil bottom, renewables top
ordered_cols = [
    "Coal", "Oil", "Natural gas",
    "Hydro", "Wind", "Solar PV", "Solar thermal", "Biofuels"
]

energy_colors = {
    "Coal": "#4D4D4D",
    "Oil": "#7F7F7F",
    "Natural gas": "#B2B2B2",
    "Hydropower": "#1f77b4",
    "Wind": "#2ca02c",
    "Solar PV": "#ffbb00",
    "Solar thermal": "#ff7f0e",
    "Biofuels": "#9467bd",
    "Biofuels and waste": "#9467bd",
    "Geothermal":"#ff3700"
}

In [ ]:
def stacked_area_plot(
    df,
    index_col,
    category_col,
    value_col,
    ordered_categories,
    ylabel,
    xlim=None,
    title=None,
    color_dict=None,
):

    pt = (
        df.pivot_table(
            index=index_col,
            columns=category_col,
            values=value_col,
            aggfunc="sum",
        )
        .reindex(columns=ordered_categories)
        .sort_index()
    )

    years = pt.index.values
    values = pt.values.T

    fig, ax = plt.subplots()

    colors = None
    if color_dict is not None:
        colors = [color_dict[c] for c in ordered_categories]

    ax.stackplot(years, values, labels=ordered_categories, colors=colors)

    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)

    if xlim:
        ax.set_xlim(*xlim)

    if title:
        ax.set_title(title)

    ax.legend(loc="upper left", ncol=2, frameon=False)

    plt.tight_layout()
    return fig, ax


## Electricity Generation
An important form of transformation is the generation of electricity. Thermal power plants generate electricity by harnessing the heat of burning fuels or nuclear reactions – during which up to half of their energy content is lost. Renewable power sources generate electricity directly from natural forces such as the sun, wind, or the movement of water. Description and data from [IEA](https://www.iea.org/countries/australia/energy-mix#how-is-energy-used-in-australia)

In [ ]:
df_elec = pd.read_csv(dd/'International Energy Agency - electricity generation in Australia.csv')
df_elec['Value'] = df_elec['Value']/1e3 # GWh -> TWh
df_elec['Value'] = df_elec['Value']*3.6e-3 # TWh -> EJ
df_elec.rename(columns={'Value': 'Electricty Generation (EJ)','electricity generation in Australia': 'Primary Source'}, inplace=True)
# df_elec.loc[df_elec['Primary Source']=="Geothermal, solar, wind, etc.",'Primary Source'] = "Renewables"

In [ ]:
df_elec

In [ ]:
electric_order = [
    "Coal",
    "Oil",
    "Natural gas",
    "Hydropower",
    "Wind",
    "Solar PV",
    "Solar thermal",
    "Biofuels",
    'Geothermal'
]

fig, ax = stacked_area_plot(
    df=df_elec,
    index_col="Year",
    category_col="Primary Source",
    value_col="Electricty Generation (EJ)",
    ordered_categories=electric_order,
    ylabel="Electricity Generation (EJ)",
    xlim=(2000, 2023),
    title="Australian Electricity Generation by Source",
    color_dict=energy_colors
)
ax.set_ylim((0, 1.2))
plt.show()


## Total energy supply. 
Total energy supply (TES) includes all the energy produced in or imported to a country, minus that which is exported or stored. It represents all the energy required to supply end users in the country. Some of these energy sources are used directly while most are transformed into fuels or electricity for final consumption. Description and data from [IEA](https://www.iea.org/countries/australia/energy-mix#how-is-energy-used-in-australia).

In [ ]:
file = dd/"International Energy Agency - total energy supply in Australia.csv"
df_total = pd.read_csv(file)
df_total['Value'] = df_total['Value']/1e6 # TJ to EJ
df_total.rename(columns={'Value': 'Energy Supply (EJ)',
                         'total energy supply in Australia': 'Primary Source'}, inplace=True)
# df_total.loc[df_total['Primary Source']=="Geothermal, solar, wind, etc.",'Primary Source'] = "Renewables"

In [ ]:
df_total

In [ ]:
primary_order =[
    'Coal and coal products',
    'Oil and oil products',
    'Natural gas',
    'Hydropower',
    'Biofuels and waste',
    "Solar, wind and other renewables"
]

primary_colors = {
    'Coal and coal products': energy_colors['Coal'],
    'Oil and oil products': energy_colors['Oil'],
    'Natural gas': energy_colors['Natural gas'],
    "Solar, wind and other renewables": "#19d119",
    "Biofuels and waste": "#9467bd",
    "Hydropower": energy_colors['Hydropower']
}

fig, ax = stacked_area_plot(
    df=df_total,
    index_col="Year",
    category_col="Primary Source",
    value_col="Energy Supply (EJ)",
    ordered_categories=primary_order,
    ylabel="Total Energy Supply (EJ)",
    xlim=(2000, 2023),
    title="Australian Primary Energy Supply by Source",
    color_dict=primary_colors
)
ax.set_ylim((0,7))
plt.show()

## Final energy consumption
Total final consumption (TFC) is the energy consumed by end users such as individuals and businesses to heat and cool buildings, to run lights, devices, and appliances, and to power vehicles, machines and factories. It also includes non-energy uses of energy products, such as fossil fuels used to make chemicals.

Some of the energy found in primary sources is lost when converting them to useable final products, especially electricity. As a result, the breakdown of final consumption can look very different from that of the primary energy supply (TES). Both are needed to fully understand the energy system.

In [ ]:
file = dd/"International Energy Agency - total final consumption in Australia by end use.csv"
df_final = pd.read_csv(file)
df_final['Value'] = df_final['Value']/1e6 # TJ to EJ
df_final.rename(columns={'Value': 'Total Final Consumption (EJ)',
                        'total final consumption in Australia': 'End Use'}, 
                inplace=True)

In [ ]:
final_order =[
    'Coal and coal products',
    'Primary oil',
    'Oil products',
    'Natural gas',
    'Heat',
    "Electricity",
    "Solar, wind and other renewables",
    "Biofuels and waste"
]

final_colors = {
    'Coal and coal products': energy_colors['Coal'],
    'Primary oil': energy_colors['Oil'],
    'Oil products': energy_colors['Oil'],
    'Natural gas': energy_colors['Natural gas'],
    'Heat': "#ff7f0e",
    "Electricity": "#424fc5",
    "Solar, wind and other renewables": "#19d119",
    "Biofuels and waste": "#9467bd"
}


fig, ax = stacked_area_plot(
    df=df_final,
    index_col="Year",
    category_col="End Use",
    value_col="Total Final Consumption (EJ)",
    ordered_categories=final_order,
    ylabel="Total Final Consumption (EJ)",
    xlim=(2000, 2023),
    title="Australian Final Energy Use",
    color_dict=final_colors
)
ax.set_ylim((0,7))
plt.show()

## Energy consumption by sector
The sectoral breakdown of a country's energy demand, which is based on its economy, geography and history, can greatly impact its energy needs and which energy sources it relies on to meet those needs – such as fueling automobiles, heating or cooling homes or running factories.

In [ ]:
file = dd/"International Energy Agency - total final consumption in Australia by sector.csv"
df_final_sector = pd.read_csv(file)
df_final_sector['Value'] = df_final_sector['Value']/1e6 # TJ to EJ
df_final_sector.rename(columns={'Value': 'Total Final Consumption (EJ)',
                        'total final consumption in Australia': 'End Use'}, 
                inplace=True)

In [ ]:
df_final_sector

In [ ]:
sector_order =[
    'Industry',
    'Transport',
    'Residential',
    'Commercial and public services',
    'Agriculture and forestry',
    'Fishing',
    'Other non-specified',
    'Non-energy use',
]

sector_colors = {
    'Industry': '#1f77b4',
    'Transport': '#ff7f0e',
    'Residential': '#2ca02c',
    'Commercial and public services': '#d62728',
    'Agriculture and forestry': '#9467bd',
    'Fishing': '#8c564b',
    'Other non-specified': '#e377c2',
    'Non-energy use': '#7f7f7f'
}


fig, ax = stacked_area_plot(
    df=df_final_sector,
    index_col="Year",
    category_col="End Use",
    value_col="Total Final Consumption (EJ)",
    ordered_categories=sector_order,
    ylabel="Total Final Consumption (EJ)",
    xlim=(2000, 2023),
    title="Australian Final Energy Use",
    color_dict=sector_colors
)
ax.set_ylim((0,7))
plt.show()